## Nome: Felipe Tagawa Reis
## Matricula: 2037
-----


# **Atividade 16 - Testes Estatísticos Caracterizados usando a Escala de Dependência de Smartphone**



## Instruções gerais

- Execute as células na ordem em que aparecem.
- Sempre **declare as hipóteses** antes de calcular (H0 = igualdade; H1 = diferença).
- Para tabelas 2×2, você pode usar:
  - `stats.chi2_contingency(obs, correction=False)` → χ² clássico;
  - `stats.chi2_contingency(obs, correction=True)` → com **Yates**;
  - `stats.fisher_exact(obs, alternative="two-sided")` → **Fisher** (n pequeno ou valores esperados < 5);
  - `mcnemar(tabela_pareada, exact=True/False, correction=True/False)` → **McNemar** (amostras pareadas).

In [30]:
# Imports úteis para toda a atividade
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option("display.precision", 4)


## Funções auxiliares


In [31]:
def esperados_from_obs(obs):
    obs = np.asarray(obs, dtype=float)
    row_sum = obs.sum(axis=1, keepdims=True)
    col_sum = obs.sum(axis=0, keepdims=True)
    total = obs.sum()
    return row_sum @ col_sum / total

def decide_from_p(p, alphas=(0.10, 0.05, 0.01)):
    return {a: ("Rejeita H0" if p < a else "Não rejeita H0") for a in alphas}



# Exercício 1 — Qui-quadrado clássico (amostras independentes, n > 40)

**Contexto:** Uma amostra de estudantes de cursos de Engenharia respondeu à *Cell Phone Dependence Scale*. Cada estudante foi classificado em "Leve", "Moderado" ou "Grave". Para uma análise 2×2, definimos:

- Dependência leve  
- Dependência moderada/grave  

Queremos verificar se o **sexo** do estudante está associado ao nível de dependência de smartphones.

A tabela de valores observados (2×2), n = 224:

| Sexo        | Dependência leve | Dependência moderada/grave | Total |
|-------------|------------------|-----------------------------|--------|
| Feminino    | 11               | 53                          | 64     |
| Masculino   | 57               | 103                         | 160    |
| **Total**   | 68               | 156                         | 224    |

1. Formule H0 e H1.  
2. Calcule a **tabela de esperados** e a estatística **χ² calculado** (sem correção).  
3. Obtenha `gl`, **χ² tabelado** para α ∈ {10%,5%,1%} (use SciPy) e o **p-valor**.  
4. Decida e conclua sobre a **associação entre sexo e dependência de smartphones**.


*H0*: Não há relação entre sexo e tipo de dependência em smartphones;

*H1*: Há relação entre sexo e tipo de dependência em smartphones.

In [32]:

# Tabela observada (Exemplo 1)
obs1 = np.array([[11, 53],
                 [57, 103]])
obs1_df = pd.DataFrame(obs1, index=["Feminino","Masculino"], columns=["Dependência leve","Dependência moderada/grave"])


In [33]:

# χ² clássico (sem correção de Yates) e esperados
chi2, p, dof, expected = stats.chi2_contingency(obs1, correction=False)
expected_df = pd.DataFrame(expected, index=obs1_df.index, columns=obs1_df.columns)
print("Os valores esperados são: ")
print(expected)
print(f"O valor calculado de χ²:{chi2}")

num_columns = len(obs1_df.columns)
num_rows = len(obs1_df.index)
gl = (num_rows - 1) * (num_columns - 1)
print(f"gl = {gl}")

# χ² tabelado para α ∈ {10%,5%,1%} (use SciPy) e o p-valor.
alphas = [0.10, 0.05, 0.01]
chi2_crit = {a: float(stats.chi2.ppf(1 - a, dof)) for a in alphas}

print(f"χ² tabelado para α ∈ {alphas} é: {chi2_crit}")

# Calcular PValor
p_value = 1 - stats.chi2.cdf(chi2, dof)
print(f"O p-valor é: {p_value}")

decide_from_p(p_value)




Os valores esperados são: 
[[ 19.42857143  44.57142857]
 [ 48.57142857 111.42857143]]
O valor calculado de χ²:7.350527903469081
gl = 1
χ² tabelado para α ∈ [0.1, 0.05, 0.01] é: {0.1: 2.705543454095404, 0.05: 3.841458820694124, 0.01: 6.6348966010212145}
O p-valor é: 0.006704306708170349


{0.1: 'Rejeita H0', 0.05: 'Rejeita H0', 0.01: 'Rejeita H0'}

**Conclusão:** Como H0 foi rejeitado em todos os casos, podemos inferir que há provas suficientes (com os dados dispostos) de que a dependência em smartphones tem relação concreta com o sexo dos estudantes, ao nível de significância de 0.67%.


## Exercício 2 — Teste Exato de Fisher (amostras independentes, n < 20 ou esperados < 5)

**Contexto:** Na mesma amostra de estudantes de Engenharia, além da classificação do nível de dependência de smartphones, foram coletadas informações sobre a participação dos pais na vida escolar.  
Queremos investigar se a **participação dos pais na vida escolar** está associada à ocorrência de **dependência grave** de smartphones.

Para isso, consideramos:
- Dois grupos de pais:
  - **Nunca** participam da vida escolar.
  - **Sempre** participam da vida escolar.
- Duas categorias de dependência:
  - **Grave**
  - **Não grave** (Leve ou Moderado).

A tabela de valores observados (2×2) é:

| Pais participam da vida escolar? | Grave | Não grave | Total |
|----------------------------------|:-----:|:---------:|:-----:|
| Nunca                            |   2   |    45     |  47   |
| Sempre                           |   9   |    69     |  78   |
| **Total**                        |  11   |   114     | 125   |

1. Formule H0 e H1 (duas caudas).
2. Aplique o **teste exato de Fisher** (two-sided) e obtenha o **p-valor**.
3. Decida em α = 5% e conclua sobre a associação entre participação dos pais e dependência grave.


*H0*: Não há relação entre Participação dos Pais e o nível de gravidade da dependência em smartphones;

*H1*: Há relação entre Participação dos Pais e o nível de gravidade da dependência em smartphones.

In [34]:

obs3 = np.array([[2,45],
                 [9,69]])
oddsratio, p_fisher = stats.fisher_exact(obs3, alternative="two-sided")
print(f"O p-valor é: {p_fisher}")

decide_from_p(p_fisher, alphas=(0.05,))


O p-valor é: 0.20611729432572512


{0.05: 'Não rejeita H0'}


**Conclusão:** Como o *H0* não foi rejeitado, podemos inferir que não há dados suficientes para afirmar que há relação entre Participação dos Pais e o Nível de Gravidade da Dependência em smartphones, ao nível de significância de 20,61%.

----
# Parte Teórica

# Responda as perguntas abaixo, utilizando suas próprias palavras!

1- O que significa dizer que duas variáveis são “independentes” no contexto de estatística?

R: Dizer que duas variáveis são "independentes" significa que uma não influencia diretamente na outra durante a análise(Não é antes e depois).


2- Explique por que não se usa correção de Yates quando a amostra é grande.

R: Quando as frequências são muito pequenas, a aproximação encontrada usando correção de Yates não é tão boa, além da aproximação do Qui-Quadrado ser boa para valores altos.


3- Por que o Teste Exato de Fisher é indicado quando n < 20 ou quando valores esperados < 5?

R: Ele é indicado porque não depende de aproximações, é um valor exato, como diz o próprio nome do teste, fazendo com que não seja necessário utilizar os outros métodos aproximados.


4- Qual é o principal motivo do Teste de Fisher ser considerado “exato”?

R: Ele é considerado exato porque ele calcula a probabilidade exata da tabela, sem aproximações.